# LLM-Guided Embedding Adaptation — Real Fine-Tune on Kaggle (GPU + OpenRouter)

Runs the full pipeline on your own data:

```
Your CSV -> sample confusing/boundary triplets -> ask a real LLM (via OpenRouter)
        "closer to B or C?" -> fine-tune all-MiniLM-L6-v2 on those answers (GPU)
        -> compare_embedders(before, after) -> did it help?
```

This is the "real" companion to `embedding_adaptation_demo.ipynb` (which proves the pipeline
works for free with a fake oracle LLM, no API key, no GPU). Here we use:

- **Your CSV** — every row, no subsampling
- **A real LLM via OpenRouter** for the triplet judgments
- **Real sentence-transformers fine-tuning** of `all-MiniLM-L6-v2` (GPU-accelerated)
- The two **bonus extras**: LLM keyphrase expansion + low-confidence correction

**Before you run this:**
1. Turn on a GPU accelerator (Settings -> Accelerator -> GPU T4 x2 or better).
2. Add your OpenRouter API key as a Kaggle Secret named `OPENROUTER_API_KEY`
   (Add-ons -> Secrets), or paste it directly in the labeler cell below.
3. Attach your CSV as a Kaggle Dataset input, or set the `CSV_PATHS` env var.

**Cost & time** (rough, for ~1000 documents / ~1000 triplets / 1 epoch):
- OpenRouter (`claude-3.5-haiku`): ~125 batched calls, well under $1.
- Fine-tuning on a T4 GPU: a few minutes. On CPU this can take 10-30+ minutes — use a GPU.

In [ ]:
# Installs GraphWeave with the LLM + adaptation extras (OpenRouter uses the `openai` package;
# real fine-tuning needs `datasets` + `accelerate`). Swap the branch for `@main` once merged.
%pip install -q "graphweave[full] @ git+https://github.com/nevil-mathew/topic-extraction-poc.git@llm-embedding-adaptation"

In [ ]:
import torch

# torch.cuda.is_available() can return True even when the assigned GPU's compute
# capability isn't supported by this PyTorch build (e.g. Kaggle's legacy P100 —
# sm_60 — vs a PyTorch wheel that only supports sm_70+). Run one tiny real op to
# catch that here rather than crashing deep inside sentence-transformers later.
DEVICE = None  # None = let sentence-transformers auto-pick; forced to "cpu" below if needed

if torch.cuda.is_available():
    try:
        torch.zeros(1, device="cuda") + 1
        print(f"GPU: {torch.cuda.get_device_name(0)} — fine-tuning will be fast.")
    except RuntimeError as e:
        DEVICE = "cpu"
        print(f"GPU detected but unusable with this PyTorch build ({e})")
        print("Falling back to CPU. In Kaggle: Settings -> Accelerator -> switch to")
        print("'GPU T4 x2' (P100's compute capability 6.0 is older than what recent")
        print("PyTorch wheels support) to actually use the GPU.")
else:
    DEVICE = "cpu"
    print("No GPU detected. Go to Settings -> Accelerator and turn on a GPU before running "
          "the fine-tuning cell below, or it will be very slow on CPU.")

## 1. Load your CSV

Same config-driven pattern as `challenges_clustering_kaggle.ipynb`: point `CSV_PATHS` at one
or more attached Kaggle datasets (or set the `CSV_PATHS`/`CSV_PATH` env var), name your
free-text column in `TEXT_COL`, and — if you have one — a ground-truth category column in
`LABEL_COL` (optional; only used later to compute ARI/NMI if you have labels to check against).

Every row is used — no subsampling.

In [ ]:
import os, glob
import numpy as np
import pandas as pd

# --- data source ---
# Point this at one or more of your own CSVs (e.g. an attached Kaggle Dataset). Accepts a list,
# or a comma-separated string via the CSV_PATHS env var.
_default_csv_paths = [
    "/kaggle/input/your-dataset/your_file.csv",
]
_env_csv_paths = os.environ.get("CSV_PATHS") or os.environ.get("CSV_PATH")
if _env_csv_paths:
    CSV_PATHS = [p.strip() for p in _env_csv_paths.split(",") if p.strip()]
else:
    CSV_PATHS = _default_csv_paths
CSV_PATHS = [p for p in CSV_PATHS if os.path.exists(p)]

if not CSV_PATHS:
    # Local fallback for running outside Kaggle
    CSV_PATHS = sorted(glob.glob(os.path.expanduser("~/Downloads/*.csv")))

if not CSV_PATHS:
    raise FileNotFoundError(
        "No CSV found. Attach your CSV as a Kaggle Dataset and update _default_csv_paths above, "
        "or set the CSV_PATHS env var."
    )

# --- column names (rename here if your CSV uses different headers) ---
TEXT_COL = "text"    # required: free-text column, one document per row
LABEL_COL = None     # optional: ground-truth category column (enables ARI/NMI later)

frames = []
for path in CSV_PATHS:
    frame = pd.read_csv(path, dtype=str)
    if TEXT_COL not in frame.columns:
        raise ValueError(f"{path}: required column '{TEXT_COL}' not found (have: {list(frame.columns)})")
    frame["source_file"] = os.path.basename(path)
    frames.append(frame)
raw = pd.concat(frames, ignore_index=True)
raw = raw[raw[TEXT_COL].notna() & (raw[TEXT_COL].str.strip() != "")].reset_index(drop=True)

documents = raw[TEXT_COL].str.strip().tolist()

if LABEL_COL and LABEL_COL in raw.columns:
    codes, categories_index = pd.factorize(raw[LABEL_COL])
    true_labels = codes
    print(f"ground-truth labels found in '{LABEL_COL}': {len(categories_index)} categories")
else:
    true_labels = None
    print("no ground-truth label column configured — ARI/NMI columns will be skipped later; "
          "everything else (held-out triplet accuracy, silhouette, coherence, forgetting check) "
          "still works without labels.")

print(f"{len(documents)} documents loaded from {len(CSV_PATHS)} CSV file(s)")

## 2. Set up the LLM (OpenRouter)

Get a key at https://openrouter.ai/keys. On Kaggle, store it as a **Secret**
(Add-ons -> Secrets -> name it `OPENROUTER_API_KEY`) rather than pasting it in plain text —
the cell below picks it up automatically if you do.

The triplet-judgment task ("is A closer to B or C?") is a simple classification call, not
something that needs a frontier model — a cheap flash/DeepSeek-class model is plenty and
often 10-50x cheaper per token. The default below (`google/gemini-2.0-flash-001`) is a good
starting point; check current pricing at https://openrouter.ai/models and swap in whichever
cheap model you prefer (`deepseek/deepseek-chat`, `meta-llama/llama-3.1-8b-instruct`,
`qwen/qwen-2.5-7b-instruct`, etc. are all reasonable alternatives). The same model is reused
for the keyphrase-expansion and low-confidence-correction cells later.

**Reasoning models:** `LLMLabeler` automatically sends OpenRouter's `reasoning: {enabled:
false}` field on every call, so even if you pick a reasoning-capable model it won't burn
its token budget on hidden chain-of-thought before answering (models that don't support
toggling this just ignore the field). Still, a plain instruct/flash model is faster and
cheaper for this task — reasoning ability adds cost and latency you don't need here.

In [ ]:
import os
from graphweave import LLMLabeler

try:
    from kaggle_secrets import UserSecretsClient
    OPENROUTER_API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")  # paste your key directly here if not using Secrets

# Fail fast with an actionable message instead of a cryptic AuthenticationError several
# cells later. On Kaggle, creating a Secret isn't enough — you must also toggle it
# "Attach"-ed to THIS notebook (Add-ons -> Secrets), then re-run this cell (restart the
# kernel if you attached it after the kernel started).
if not OPENROUTER_API_KEY or not OPENROUTER_API_KEY.startswith("sk-or-"):
    raise ValueError(
        "OPENROUTER_API_KEY is missing or doesn't look like a real OpenRouter key "
        "(should start with 'sk-or-'). On Kaggle: Add-ons -> Secrets -> create a secret "
        "named exactly 'OPENROUTER_API_KEY' -> toggle it ATTACHED to this notebook -> "
        "re-run this cell (restart the kernel if it still fails). Or paste your key "
        "directly in place of the os.environ.get(...) call above for a quick local test."
    )

OPENROUTER_MODEL = "google/gemini-2.0-flash-001"  # cheap, fast, plenty for B-or-C triplet judgments

labeler = LLMLabeler(
    provider="openrouter",
    api_key=OPENROUTER_API_KEY,
    model=OPENROUTER_MODEL,
    verbose=True,
)
print(f"LLMLabeler ready: {OPENROUTER_MODEL} via OpenRouter")

## 3. Embed with all-MiniLM-L6-v2

Real sentence-transformers embeddings this time (not the LSA fallback from the free demo
notebook) — `SentenceTransformer` auto-detects the GPU, so this is fast on Kaggle.

In [ ]:
from graphweave import EmbeddingEngine

engine = EmbeddingEngine(model_name="all-MiniLM-L6-v2", batch_size=128, show_progress=True, device=DEVICE)
baseline_embeddings = engine.encode(documents)
print("baseline_embeddings shape:", baseline_embeddings.shape)

## 4. Fit the baseline model

In [ ]:
from graphweave import GraphWeave, GraphWeaveConfig

baseline_config = GraphWeaveConfig(
    embedding_model="all-MiniLM-L6-v2",  # must match the embedder above — adapt_and_refit
                                          # reads this to know what to fine-tune later
    min_cluster_fraction=0.005,          # scales min_cluster_size with corpus size (recommended)
    random_state=42,
    verbose=True,
)

model = GraphWeave(config=baseline_config)
model.fit(documents, embeddings=baseline_embeddings)

n_topics = len([t for t in model.topics_ if t.topic_id != -1])
print(f"Baseline: {n_topics} topics, {np.mean(model.labels_ == -1):.1%} outliers")
model.get_topic_info()[lambda d: d.Topic != -1].sort_values("Size", ascending=False).head(10)

## 5. Real LLM-guided fine-tuning (the main event)

Same call as the free demo notebook, but:
- `labeler` is the real OpenRouter `LLMLabeler` from Section 2 (not an oracle)
- `adapter_mode="finetune"` — a real sentence-transformers fine-tune of `all-MiniLM-L6-v2`
  (`MultipleNegativesRankingLoss`, 1 epoch, low learning rate), not the numpy linear adapter

What happens under the hood:
1. Sample `n_triplets` (anchor, B, C) triplets — anchors are the documents the model's soft
   assignment is least confident about (highest entropy), since those boundary cases benefit
   most from an LLM's judgment.
2. Ask OpenRouter, batched, "is A closer to B or C?" (~8 triplets/call). ~20% are held out
   and never used for training, so we can honestly measure whether it helped afterward.
3. Fine-tune `all-MiniLM-L6-v2` on the LLM-judged (anchor, positive, negative) triples.
4. Refit GraphWeave on the adapted embeddings — `new_model` is fully independent; `model`
   (the baseline) is left untouched, so you always have both to compare.

In [ ]:
from graphweave.adaptation import AdaptationConfig, adapt_and_refit

adapt_config = AdaptationConfig(
    adapter_mode="finetune",     # real sentence-transformers fine-tune (GPU-accelerated)
    n_triplets=1000,             # ~ ClusterLLM's budget; raise for a bigger/harder corpus
    triplet_sampling="entropy",  # focus the LLM's budget on the least-confident documents
    holdout_frac=0.2,
    llm_batch_size=8,
    # llm_max_tokens=2000,        # extra safety valve: raise this if a reasoning model's
                                   # thinking somehow still gets through despite Section 2's
                                   # auto-disable (rare) and you see "finish_reason=length"
    epochs=1,
    learning_rate=2e-5,
    train_batch_size=64 if DEVICE != "cpu" else 32,  # GPU can take a bigger batch than CPU
    device=DEVICE,
    cache_path="/kaggle/working/adaptation_triplet_cache.jsonl",  # re-runs pay $0 for cached triplets
    random_state=42,
    verbose=True,
)

new_model, report = adapt_and_refit(
    model, labeler, config=adapt_config, evaluate=True, labels_true=true_labels
)

print(f"\nLLM calls    : {report['n_llm_calls']} (cache hits: {report['n_cache_hits']}, unparsed: {report['n_unparsed']})")
print(f"Triplets     : {report['n_train_triplets']} train / {report['n_holdout_triplets']} holdout")
print(f"Held-out acc : {report['holdout_triplet_acc_before']:.3f} -> {report['holdout_triplet_acc_after']:.3f}")
if "forgetting" in report:
    fg = report["forgetting"]
    print(f"Forgetting   : generic-STS correlation {fg['spearman_before']:.3f} -> {fg['spearman_after']:.3f} "
          f"(delta {fg['delta']:+.3f}; a drop below -0.05 would be a red flag)")

report["comparison"]

## 6. Did it actually help?

**If your CSV has no ground-truth labels** (the common case — real data usually doesn't),
judge the adaptation with the signals that don't need labels, in this order of trust:

1. **`holdout_triplet_acc`** — did the embeddings move toward the LLM's judgments on
   triplets it never trained on? This is the most direct, corpus-specific signal.
2. **Forgetting check** — did general semantic competence regress? A small negative delta
   is fine; a large one (below -0.05) means back off (fewer epochs / lower learning rate).
3. **Intrinsic clustering metrics** — `silhouette` (higher better), `davies_bouldin`
   (lower better), `stability` (higher better), `coherence_mean` (higher better).

**If you do have labels** (`LABEL_COL` set above), `ari`/`nmi`/`cluster_accuracy` are the most
direct ground-truth signal — but treat them as a bonus check here, not the primary one.

`adapt_and_refit` already warns automatically if held-out triplet accuracy didn't improve —
watch for that warning above.

In [ ]:
df = report["comparison"]
base = df[df.variant == "baseline"].iloc[0]
adapted = df[df.variant == "adapted"].iloc[0]

higher_is_better = {
    "silhouette": True, "davies_bouldin": False, "calinski_harabasz": True,
    "stability": True, "coherence_mean": True, "holdout_triplet_acc": True,
}

print(f"{'metric':22s} {'baseline':>10s} {'adapted':>10s}   verdict")
print("-" * 60)
for col, higher_better in higher_is_better.items():
    if col not in df.columns:
        continue
    delta = adapted[col] - base[col]
    improved = delta > 0 if higher_better else delta < 0
    verdict = "better" if improved else ("worse" if abs(delta) > 1e-9 else "unchanged")
    print(f"{col:22s} {base[col]:10.3f} {adapted[col]:10.3f}   {verdict}")

if "ari" in df.columns:
    print(f"\nGround truth ARI     : {base['ari']:.3f} -> {adapted['ari']:.3f}")
    print(f"Ground truth NMI     : {base['nmi']:.3f} -> {adapted['nmi']:.3f}")
    print(f"Cluster accuracy     : {base['cluster_accuracy']:.3f} -> {adapted['cluster_accuracy']:.3f}")

## 7. Save the fine-tuned model

Kaggle sessions are ephemeral — save the adapted embedder to `/kaggle/working` (download it
from the Output panel, or push it as a new Kaggle Dataset) so you can reuse it without paying
for another fine-tuning run.

In [ ]:
adapter = report["adapter"]
adapter.save("/kaggle/working/adapted_embedder")
print("Saved to /kaggle/working/adapted_embedder — download this folder from the Output panel.")

# Reload and use later (no need to re-fine-tune):
# from graphweave.adaptation import EmbeddingAdapter
# reloaded = EmbeddingAdapter.load("/kaggle/working/adapted_embedder")
# fresh_embeddings = reloaded.encode(new_documents)

## 8. Bonus extra #1: LLM keyphrase expansion

An independent, no-fine-tuning-needed lever from Viswanathan et al. (TACL 2024): ask the LLM
for a handful of keyphrases per document, then blend their embedding into the document's own.
Useful on its own, or combined with the fine-tuned embedder above.

In [ ]:
from graphweave.adaptation import compare_embedders, generate_keyphrases, keyphrase_expand_embeddings

keyphrases = generate_keyphrases(
    labeler, documents, n_keyphrases=5,
    cache_path="/kaggle/working/keyphrase_cache.jsonl",
)
print("Example:", documents[0][:120], "->", keyphrases[0])

keyphrase_embeddings = keyphrase_expand_embeddings(documents, keyphrases, engine, weight=0.5)
print("keyphrase_embeddings shape:", keyphrase_embeddings.shape)

kw_df = compare_embedders(
    documents,
    {"baseline": baseline_embeddings, "keyphrase_expanded": keyphrase_embeddings},
    labels_true=true_labels,
    base_config=baseline_config,
)
kw_df

## 9. Bonus extra #2: post-hoc low-confidence correction

Rather than touching every document, ask the LLM to re-adjudicate only the ones the model
itself is least sure about (a small soft-assignment margin between its top-2 candidate
topics). Run with `dry_run=True` first to preview, then apply in a separate cell.

In [ ]:
from graphweave.adaptation import reassign_low_confidence

preview = reassign_low_confidence(
    new_model, labeler, margin_threshold=0.15, max_docs=200, dry_run=True,
)
print(f"{len(preview)} low-confidence documents reviewed; "
      f"{(preview.new_topic != preview.old_topic).sum()} would be reassigned")
preview.head(10)

In [ ]:
# Re-run for real once you're happy with the preview above (mutates new_model.labels_ in place):
applied = reassign_low_confidence(
    new_model, labeler, margin_threshold=0.15, max_docs=200, dry_run=False,
)
print(f"Applied {applied['applied'].sum()} reassignments out of {len(applied)} reviewed")

## 10. Summary

| Step | What it does |
|---|---|
| Load CSV | Every row from your dataset, no subsampling |
| Embed | `all-MiniLM-L6-v2` via `EmbeddingEngine` (GPU-accelerated) |
| Baseline fit | Standard `GraphWeave.fit()` |
| `adapt_and_refit(model, labeler, config=AdaptationConfig(adapter_mode="finetune"))` | Real LLM triplet judgments (OpenRouter) -> real sentence-transformers fine-tune -> refit |
| `report["comparison"]` | Baseline vs. adapted, side by side |
| `report["adapter"].save(path)` | Persist the fine-tuned model for reuse |
| `generate_keyphrases` + `keyphrase_expand_embeddings` | Bonus: no-fine-tune embedding boost |
| `reassign_low_confidence` | Bonus: post-hoc correction of only the least-confident assignments |

See `notebooks/embedding_adaptation_demo.ipynb` for a $0, no-API-key version of Sections 1-6
(useful for testing the pipeline itself before spending real LLM/GPU budget), and the
README's [*LLM-Guided Embedding Adaptation*](../README.md#llm-guided-embedding-adaptation)
section for the full reference.